# VSC22 Video Copy Detection Pipeline

This notebook runs the full inference pipeline:
1. Environment verification
2. Download pretrained checkpoints
3. Download VCDB sample data
4. Prepare data (convert to VSC22 format)
5. Descriptor Track: extract features + retrieval
6. Matching Track: segment-level detection
7. Visualize results

## 1. Environment Verification

In [ ]:
import os
from pathlib import Path
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

import torch
import torchvision
import numpy as np
import pandas as pd
import sklearn
import timm
import PIL
import sys

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"Torchvision:  {torchvision.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"timm:         {timm.__version__}")
print(f"Pillow:       {PIL.__version__}")
print(f"CUDA:         {torch.version.cuda or 'N/A'}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Device:       CUDA - {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device("cpu")  # MPS has TorchScript issues, default to CPU
    print(f"Device:       CPU (MPS available but using CPU for TorchScript compat)")
else:
    DEVICE = torch.device("cpu")
    print(f"Device:       CPU only")

try:
    import faiss
    print(f"FAISS:        installed (GPUs: {faiss.get_num_gpus()})")
except ImportError:
    print("FAISS:        NOT INSTALLED")

# Check ffmpeg
ffmpeg_ok = os.system("ffmpeg -version > /dev/null 2>&1") == 0
print(f"ffmpeg:       {'OK' if ffmpeg_ok else 'NOT FOUND'}")

_here = Path.cwd().resolve()
REPO_ROOT = _here
for _p in [_here, *_here.parents]:
    if (_p / "VSC22-Descriptor-Track-1st").is_dir():
        REPO_ROOT = _p
        break
print(f"\nRepo root:    {REPO_ROOT}")

## 2. Download & Verify Pretrained Checkpoints

In [ ]:
import subprocess

CKPT_DIR = os.path.join(REPO_ROOT, "VSC22-Descriptor-Track-1st", "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

expected_files = [
    "swinv2_v115.torchscript.pt", "swinv2_v107.torchscript.pt",
    "swinv2_v106.torchscript.pt", "vit_v68.torchscript.pt",
    "clip.torchscript.pt", "vsm.torchscript.pt", "pca_model.pkl",
    "submit_cls_model1.pt", "submit_cls_model2.pt",
    "submit_match_model1.pt", "submit_match_model2.pt",
]

existing = [f for f in expected_files if os.path.exists(os.path.join(CKPT_DIR, f))]
missing = [f for f in expected_files if f not in existing]

if missing:
    print(f"Missing {len(missing)} checkpoint files. Downloading...")
    !cd {CKPT_DIR} && gdown 1GL0xhTTSHav_iG79yJ1jqgQcmuJFs_lF -O ../checkpoints.tar.gz
    !cd {CKPT_DIR} && tar -xzf ../checkpoints.tar.gz -C . && rm -f ../checkpoints.tar.gz
    # Check if files ended up in a subdirectory
    for f in expected_files:
        if not os.path.exists(os.path.join(CKPT_DIR, f)):
            # Try to find it recursively
            for root, dirs, files in os.walk(CKPT_DIR):
                if f in files:
                    src = os.path.join(root, f)
                    os.rename(src, os.path.join(CKPT_DIR, f))
                    break
else:
    print(f"All {len(expected_files)} checkpoint files present.")

print("\nCheckpoint directory contents:")
for f in sorted(os.listdir(CKPT_DIR)):
    size_mb = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6
    print(f"  {f:40s} {size_mb:8.1f} MB")

In [ ]:
# Verify TorchScript models load correctly
print(f"Loading TorchScript models on {DEVICE}...")
torchscript_models = ["vit_v68", "swinv2_v115", "swinv2_v107", "swinv2_v106", "clip", "vsm"]
for name in torchscript_models:
    path = os.path.join(CKPT_DIR, f"{name}.torchscript.pt")
    if os.path.exists(path):
        m = torch.jit.load(path, map_location=DEVICE)
        print(f"  {name}: OK")
        del m
    else:
        print(f"  {name}: FILE MISSING")

# Check matching models
for name in ["submit_cls_model1", "submit_cls_model2", "submit_match_model1", "submit_match_model2"]:
    path = os.path.join(CKPT_DIR, f"{name}.pt")
    if os.path.exists(path):
        m = torch.jit.load(path, map_location=DEVICE)
        print(f"  {name}: OK")
        del m
    else:
        print(f"  {name}: FILE MISSING (needed for Matching Track)")

# Check PCA model
import pickle
pca_path = os.path.join(CKPT_DIR, "pca_model.pkl")
if os.path.exists(pca_path):
    with open(pca_path, "rb") as f:
        pca = pickle.load(f)
    print(f"  pca_model.pkl: OK (n_components={pca.n_components_})")
    del pca
else:
    print(f"  pca_model.pkl: FILE MISSING")

print("\nCheckpoint verification complete.")

## 3. Download VCDB Sample Data

In [ ]:
VCDB_DIR = os.path.join(REPO_ROOT, "vcdb_core")

if os.path.isdir(VCDB_DIR) and len(os.listdir(VCDB_DIR)) > 0:
    print(f"VCDB data already present at {VCDB_DIR}")
    subdirs = [d for d in os.listdir(VCDB_DIR) if os.path.isdir(os.path.join(VCDB_DIR, d))]
    print(f"  {len(subdirs)} query groups found")
else:
    print("Downloading VCDB core dataset (~7GB)...")
    print("This may take a while depending on your connection.")
    !gdown --folder 0B-b0CY525pH8NjdxbFNGY0JJdGs -O {VCDB_DIR}
    print("Download complete.")

## 4. Prepare Data (Convert to VSC22 Format)

This converts VCDB videos into the format expected by the pipeline:
- Standardized video IDs (Q000001, R000001, ...)
- Metadata CSVs
- Frame extraction at 1fps into zip files

Use `--max_videos 6` for quick local testing on CPU.

In [ ]:
# Set MAX_VIDEOS=6 for local CPU testing, or None for full dataset on GPU server
MAX_VIDEOS = 6 if DEVICE.type == "cpu" else None

cmd = f"python {os.path.join(REPO_ROOT, 'prepare_data.py')} --vcdb_dir {VCDB_DIR}"
if MAX_VIDEOS:
    cmd += f" --max_videos {MAX_VIDEOS}"
    print(f"Using subset of {MAX_VIDEOS} videos for local testing")
else:
    print("Using full VCDB dataset")

print(f"Running: {cmd}")
!{cmd}

In [ ]:
# Verify data preparation
import json

data_root = os.path.join(REPO_ROOT, "VSC22-Descriptor-Track-1st", "data")
mapping_file = os.path.join(REPO_ROOT, "VSC22-Descriptor-Track-1st", "id_mapping.json")

if os.path.exists(mapping_file):
    with open(mapping_file) as f:
        id_map = json.load(f)
    refs = {k: v for k, v in id_map.items() if k.startswith("R")}
    queries = {k: v for k, v in id_map.items() if k.startswith("Q")}
    print(f"ID mapping: {len(refs)} references, {len(queries)} queries")
    print("\nSample mappings:")
    for k, v in list(id_map.items())[:4]:
        print(f"  {k} -> {os.path.basename(v)}")

# Check metadata files
meta_dir = os.path.join(data_root, "meta", "test")
for fname in ["test_query_metadata.csv", "test_reference_metadata.csv", "test_ref_vids.txt"]:
    fpath = os.path.join(meta_dir, fname)
    if os.path.exists(fpath):
        if fname.endswith(".csv"):
            df = pd.read_csv(fpath)
            print(f"\n{fname}: {len(df)} rows")
            print(df.head(3))
        else:
            with open(fpath) as f:
                lines = f.read().strip().split("\n")
            print(f"\n{fname}: {len(lines)} entries")
    else:
        print(f"\nMISSING: {fpath}")

# Check jpg_zips
zip_dir = os.path.join(data_root, "jpg_zips")
zip_count = sum(1 for _, _, files in os.walk(zip_dir) for f in files if f.endswith(".zip"))
print(f"\njpg_zips: {zip_count} zip files")

## 5. Descriptor Track: Extract Reference Features

In [ ]:
infer_dir = os.path.join(REPO_ROOT, "VSC22-Descriptor-Track-1st", "infer")

print("=== Extracting reference features (4 models) + PCA + score normalization ===")
print("This may take a while on CPU...")
!cd {infer_dir} && bash infer_ref.sh

In [ ]:
# Move train_refs.npz to data/ (needed by query feature extraction)
import shutil

src = os.path.join(infer_dir, "outputs", "train_refs.npz")
dst_dir = os.path.join(infer_dir, "data")
os.makedirs(dst_dir, exist_ok=True)
dst = os.path.join(dst_dir, "train_refs.npz")

if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Copied train_refs.npz to {dst}")
else:
    print(f"WARNING: {src} not found. Check infer_ref.sh output above.")

# Verify outputs
outputs_dir = os.path.join(infer_dir, "outputs")
for f in sorted(os.listdir(outputs_dir)):
    fpath = os.path.join(outputs_dir, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f:40s} {size_mb:8.2f} MB")
    elif os.path.isdir(fpath):
        print(f"  {f}/")

## 6. Descriptor Track: Extract Query Features

In [ ]:
print("=== Extracting query features ===")
!cd {infer_dir} && PYTHONPATH=$PYTHONPATH:./ python3 extract_query_feats.py --split test

## 7. Descriptor Track: Retrieval (Find Matching Videos)

In [ ]:
print("=== Running retrieval (query vs reference) ===")
!cd {infer_dir} && PYTHONPATH=$PYTHONPATH:./ bash eval.sh

# Show results
desc_output = os.path.join(infer_dir, "outputs")
for f in sorted(os.listdir(desc_output)):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(desc_output, f))
        print(f"\n{f}: {len(df)} rows")
        if len(df) > 0:
            print(df.head(10))

## 8. Matching Track: Segment-Level Detection

In [ ]:
match_infer_dir = os.path.join(REPO_ROOT, "VSC22-Matching-Track-1st", "infer")

print("=== Running matching track inference ===")
!cd {match_infer_dir} && PYTHONPATH=$PYTHONPATH:./ python3 infer_matching.py --split test

## 9. View Results

In [ ]:
# Load matching results
matching_output = os.path.join(REPO_ROOT, "VSC22-Descriptor-Track-1st", "infer", 
                               "outputs", "matching", "test_matching.csv")

if os.path.exists(matching_output):
    results_df = pd.read_csv(matching_output)
    print(f"Matching results: {len(results_df)} segment pairs found")
    print(f"\nColumns: {list(results_df.columns)}")
    
    if len(results_df) > 0:
        print("\nTop matches by score:")
        print(results_df.sort_values("score", ascending=False).head(20).to_string(index=False))
        
        # Map back to original filenames
        if os.path.exists(mapping_file):
            with open(mapping_file) as f:
                id_map = json.load(f)
            
            print("\n\nResults mapped to original filenames:")
            for _, row in results_df.sort_values("score", ascending=False).head(10).iterrows():
                q_name = os.path.basename(id_map.get(row["query_id"], row["query_id"]))
                r_name = os.path.basename(id_map.get(row["ref_id"], row["ref_id"]))
                print(f"  {q_name} [{row['query_start']:.1f}s-{row['query_end']:.1f}s] "
                      f"<-> {r_name} [{row['ref_start']:.1f}s-{row['ref_end']:.1f}s] "
                      f"(score: {row['score']:.4f})")
    else:
        print("No matching segments found.")
else:
    print(f"Matching output not found at {matching_output}")
    print("Check the Matching Track inference output above for errors.")

In [ ]:
print("\n=== Pipeline Complete ===")
print(f"Device used: {DEVICE}")
print(f"\nOutputs:")
print(f"  Descriptor results:  {os.path.join(infer_dir, 'outputs')}")
print(f"  Matching results:    {matching_output}")
print(f"  ID mapping:          {mapping_file}")